# List i dict comprehension w Pythonie — od podstaw do pracy z DataFrame

Notebook krok po kroku: składnia, warianty, wydajność, a w drugiej połowie —
zastosowania typowe dla pracy analityka danych (nazwy kolumn, mapowania,
budowanie wyrażeń dla Polars, wczytywanie wielu plików).

Jedna rzecz od razu na wstępie, bo to najważniejsza praktyczna uwaga w tym
całym temacie: **comprehension to wciąż pętla** — czytelniejsza i zwykle
szybsza niż `for` z `.append()`, ale to nie to samo co wektoryzacja w
Pandas/Polars/NumPy. Sekcja 12 pokazuje to na konkretnym pomiarze czasu.

In [1]:
import pandas as pd
import polars as pl
import numpy as np
import timeit

## 1. Podstawowa składnia — pętla `for` vs list comprehension

List comprehension to skrócony zapis pętli, która buduje nową listę.

In [2]:
# pętla klasyczna
liczby = [1, 2, 3, 4, 5]
kwadraty = []
for x in liczby:
    kwadraty.append(x ** 2)
print(kwadraty)

[1, 4, 9, 16, 25]


In [3]:
# to samo jako list comprehension: [wyrażenie for element in iterowalny]
kwadraty = [x ** 2 for x in liczby]
print(kwadraty)

[1, 4, 9, 16, 25]


Schemat ogólny: `[wyrażenie for zmienna in iterowalny]`. `wyrażenie` może
być dowolnie złożone (wywołanie funkcji, `f-string`, wynik operacji) —
liczy się tylko, że dla każdego elementu zwraca jedną wartość.

In [4]:
nazwy = ["warszawa", "krakow", "gdansk"]
tytulowe = [nazwa.title() for nazwa in nazwy]
print(tytulowe)

['Warszawa', 'Krakow', 'Gdansk']


## 2. Warunki w comprehension — filtrowanie vs wyrażenie warunkowe

To dwie różne rzeczy, mylone przez początkujących — różni je **pozycja**
`if` w zapisie.

### a) `if` na końcu = filtr (pomija elementy)

In [5]:
liczby = list(range(10))

parzyste = [x for x in liczby if x % 2 == 0]   # 'if' PO 'for' -> filtruje, ile elementów wejdzie do wyniku
print(parzyste)

[0, 2, 4, 6, 8]


### b) `if ... else` przed `for` = wyrażenie warunkowe (transformuje, nie filtruje)

Długość wyniku jest zawsze taka sama jak długość wejścia — każdy element
dostaje jedną z dwóch wartości.

In [6]:
etykiety = ["parzysta" if x % 2 == 0 else "nieparzysta" for x in liczby]
print(etykiety)
print(len(liczby), len(etykiety))   # te same długości — w przeciwieństwie do filtra powyżej

['parzysta', 'nieparzysta', 'parzysta', 'nieparzysta', 'parzysta', 'nieparzysta', 'parzysta', 'nieparzysta', 'parzysta', 'nieparzysta']
10 10


Oba można połączyć — filtr na końcu, wyrażenie warunkowe na początku:

In [7]:
wynik = [x ** 2 if x % 2 == 0 else -x for x in liczby if x > 2]
print(wynik)

[-3, 16, -5, 36, -7, 64, -9]


## 3. Zagnieżdżone pętle w comprehension — spłaszczanie list list

Kolejność `for` w comprehension jest taka sama jak w zagnieżdżonych pętlach
klasycznych — zewnętrzna pętla pierwsza.

In [8]:
lista_list = [[1, 2, 3], [4, 5], [6, 7, 8, 9]]

# klasyczna pętla zagnieżdżona
splaszczona = []
for podlista in lista_list:
    for element in podlista:
        splaszczona.append(element)
print(splaszczona)

[1, 2, 3, 4, 5, 6, 7, 8, 9]


In [9]:
# to samo jako comprehension — kolejność 'for' identyczna jak w pętli klasycznej powyżej
splaszczona = [element for podlista in lista_list for element in podlista]
print(splaszczona)

[1, 2, 3, 4, 5, 6, 7, 8, 9]


Praktyczny przykład: spłaszczenie listy list nazw kolumn z kilku różnych
źródeł (np. przed budową wspólnego schematu).

In [10]:
kolumny_z_zrodel = [["id", "nazwa"], ["id", "miasto", "kraj"], ["id", "data_utworzenia"]]
wszystkie_unikalne = sorted({kolumna for zrodlo in kolumny_z_zrodel for kolumna in zrodlo})
print(wszystkie_unikalne)

['data_utworzenia', 'id', 'kraj', 'miasto', 'nazwa']


## 4. Dict comprehension — `{klucz: wartość for ... }`

Identyczna logika co list comprehension, tylko zamiast jednej wartości
budujesz parę klucz-wartość.

In [11]:
miasta = ["Warszawa", "Krakow", "Gdansk"]
dlugosci = {miasto: len(miasto) for miasto in miasta}
print(dlugosci)

{'Warszawa': 8, 'Krakow': 6, 'Gdansk': 6}


In [12]:
# dict comprehension z filtrem
ceny = {"jablko": 3.5, "banan": 5.2, "gruszka": 4.1, "winogrona": 12.0}
tanie = {produkt: cena for produkt, cena in ceny.items() if cena < 5}
print(tanie)

{'jablko': 3.5, 'gruszka': 4.1}


In [13]:
# odwrócenie słownika (klucz <-> wartość) — częsty jednowierszowiec
kod_kraju = {"Polska": "PL", "Niemcy": "DE", "Francja": "FR"}
nazwa_z_kodu = {kod: kraj for kraj, kod in kod_kraju.items()}
print(nazwa_z_kodu)

{'PL': 'Polska', 'DE': 'Niemcy', 'FR': 'Francja'}


## 5. Set comprehension — `{wartość for ...}`

Jak list comprehension, ale wynik to `set` (unikalne wartości, bez
kolejności). Przydatne do szybkiego wyciągnięcia unikalnych wartości przy
okazji transformacji — bez osobnego wywołania `set(...)` na już zbudowanej liście.

In [14]:
zamowienia = ["A", "B", "A", "C", "B", "A"]
unikalne_wielkimi = {z.lower() for z in zamowienia}
print(unikalne_wielkimi)

{'c', 'a', 'b'}


## 6. Generator expression — ta sama składnia, bez nawiasów kwadratowych

`(wyrażenie for ... )` zamiast `[wyrażenie for ...]` — wygląda niemal
identycznie, ale **nie buduje listy w pamięci**. Wartości są liczone "na
żądanie", jedna po drugiej. Kluczowe przy dużych zbiorach danych, gdy
potrzebujesz tylko zagregowanego wyniku (`sum`, `any`, `max`), a nie samej
listy pośredniej.

In [15]:
# list comprehension: buduje całą listę 10 mln liczb w pamięci, potem sumuje
%timeit sum([x for x in range(10_000_000)])

463 ms ± 7.99 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [16]:
# generator expression: liczy sumę w locie, bez trzymania całej listy w pamięci — szybsze i lżejsze
%timeit sum(x for x in range(10_000_000))

281 ms ± 10.9 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


Zasada praktyczna: jeśli wynik przekazujesz od razu do funkcji, która i tak
"skonsumuje" całą sekwencję po kolei (`sum`, `any`, `all`, `max`, `min`,
`"".join(...)`, `pd.concat(...)`), użyj generatora — nawiasy okrągłe zamiast
kwadratowych, bez zmiany reszty kodu. Jeśli potrzebujesz listy wielokrotnie
albo indeksowania po niej — zostań przy `[...]`.

## 7. Comprehension z `enumerate` i `zip`

Bardzo częste połączenie w praktyce — numerowanie elementów albo łączenie
dwóch równoległych list.

In [17]:
kolumny = ["id", "nazwa", "miasto"]

# enumerate -> para (indeks, element)
z_numerami = [f"{i}: {kolumna}" for i, kolumna in enumerate(kolumny)]
print(z_numerami)

['0: id', '1: nazwa', '2: miasto']


In [18]:
# zip -> łączenie dwóch list element po elemencie, np. budowa słownika mapującego stare -> nowe nazwy
stare_nazwy = ["ID", "NAZWA", "MIASTO"]
nowe_nazwy = ["id", "nazwa", "miasto"]

mapowanie = {stara: nowa for stara, nowa in zip(stare_nazwy, nowe_nazwy)}
print(mapowanie)

{'ID': 'id', 'NAZWA': 'nazwa', 'MIASTO': 'miasto'}


## 8. Operator morsa `:=` w comprehension (Python 3.8+)

Przydaje się, gdy warunek filtrujący i wartość w wyniku zależą od tego
samego, kosztownego obliczenia — bez `:=` musiałbyś je policzyć dwukrotnie
(raz w `if`, raz w wyrażeniu wynikowym).

In [19]:
def przelicz_kosztownie(x):
    # symulacja czegoś droższego niż zwykłe mnożenie (np. wywołanie API, parsowanie)
    return x ** 2 - 10


liczby = range(10)

# BEZ walrusa: przelicz_kosztownie() wywołane DWUKROTNIE dla każdego elementu, który przejdzie filtr
wynik_bez_walrusa = [przelicz_kosztownie(x) for x in liczby if przelicz_kosztownie(x) > 0]

# Z walrusem: policzone raz, przypisane do 'wartosc', użyte i w filtrze, i w wyniku
wynik_z_walrusem = [wartosc for x in liczby if (wartosc := przelicz_kosztownie(x)) > 0]

print(wynik_bez_walrusa)
print(wynik_z_walrusem)
assert wynik_bez_walrusa == wynik_z_walrusem

[6, 15, 26, 39, 54, 71]
[6, 15, 26, 39, 54, 71]


## 9. Praca z kolumnami DataFrame — selekcja i filtrowanie nazw

Od tego miejsca — zastosowania typowe dla analityka danych. `df.columns`
to zwykła sekwencja stringów, więc wszystko z sekcji 1-2 stosuje się wprost.

In [20]:
df = pd.DataFrame({
    "klient_id": [1, 2, 3],
    "nazwa_klienta": ["Firma A", "Firma B", "Firma C"],
    "sprzedaz_2024": [100, 200, 150],
    "sprzedaz_2025": [120, 210, 180],
    "flaga_aktywny": [True, False, True],
})

# kolumny zaczynające się od "sprzedaz_" — częste przy danych w formacie "szerokim" (rok jako sufiks)
kolumny_sprzedazy = [kol for kol in df.columns if kol.startswith("sprzedaz_")]
print(kolumny_sprzedazy)

['sprzedaz_2024', 'sprzedaz_2025']


In [21]:
# kolumny numeryczne / tekstowe — filtr po typie danych zamiast po nazwie
kolumny_numeryczne = [kol for kol in df.columns if pd.api.types.is_numeric_dtype(df[kol])]
kolumny_tekstowe = [kol for kol in df.columns if pd.api.types.is_object_dtype(df[kol])]

print("numeryczne:", kolumny_numeryczne)
print("tekstowe:", kolumny_tekstowe)

numeryczne: ['klient_id', 'sprzedaz_2024', 'sprzedaz_2025', 'flaga_aktywny']
tekstowe: []


Ten sam wzorzec w Polars — `df.columns` i `df.schema` (dict kolumna→typ)
też są zwykłymi obiektami Pythona, więc comprehension działa identycznie.

In [22]:
df_pl = pl.from_pandas(df)

kolumny_numeryczne_pl = [kol for kol, typ in df_pl.schema.items() if typ.is_numeric()]
print(kolumny_numeryczne_pl)

['klient_id', 'sprzedaz_2024', 'sprzedaz_2025']


## 10. Budowanie słownika do `rename` / mapowania kolumn

Dict comprehension jest naturalnym sposobem budowania słownika, którego
oczekuje `df.rename(columns=...)` — zwłaszcza gdy transformacja nazwy jest
regularna (nie trzeba wypisywać każdej pary ręcznie).

In [23]:
df_brudny = pd.DataFrame(columns=["ID Klienta", "Nazwa Firmy", "Data Rejestracji"])

# ujednolicenie nazw: małe litery, spacje -> podkreślniki — reguła, nie lista ręcznych par
mapowanie_nazw = {
    kol: kol.lower().replace(" ", "_")
    for kol in df_brudny.columns
}
print(mapowanie_nazw)

df_czysty = df_brudny.rename(columns=mapowanie_nazw)
print(list(df_czysty.columns))

{'ID Klienta': 'id_klienta', 'Nazwa Firmy': 'nazwa_firmy', 'Data Rejestracji': 'data_rejestracji'}
['id_klienta', 'nazwa_firmy', 'data_rejestracji']


Podobnie — budowanie słownika `dtype` do jawnego rzutowania typów przy
wczytywaniu (`pd.read_csv(..., dtype=...)`), gdy reguła jest regularna
(np. wszystkie kolumny z prefiksem `id_` mają być `str`, żeby nie zgubić
zer wiodących).

In [24]:
wszystkie_kolumny = ["id_klienta", "id_produktu", "ilosc", "cena"]

typy = {
    kol: "str" if kol.startswith("id_") else "float64"
    for kol in wszystkie_kolumny
}
print(typy)

{'id_klienta': 'str', 'id_produktu': 'str', 'ilosc': 'float64', 'cena': 'float64'}


## 11. Budowanie listy wyrażeń agregujących dla Polars

To jeden z najbardziej praktycznych wzorców dla analityka pracującego w
Polars: `select`/`agg`/`with_columns` przyjmują **listę wyrażeń** — a listę
łatwo zbudować comprehension zamiast wypisywać każdą kolumnę ręcznie.

In [25]:
df_sprzedaz = pl.DataFrame({
    "region": ["Wschod", "Wschod", "Zachod", "Zachod"],
    "sprzedaz_styczen": [100, 150, 200, 130],
    "sprzedaz_luty": [110, 140, 210, 125],
    "sprzedaz_marzec": [120, 160, 190, 145],
})

kolumny_miesiecy = [kol for kol in df_sprzedaz.columns if kol.startswith("sprzedaz_")]

# zamiast: df.select([pl.col("sprzedaz_styczen").sum(), pl.col("sprzedaz_luty").sum(), ...])
wyrazenia_sum = [pl.col(kol).sum().alias(f"suma_{kol}") for kol in kolumny_miesiecy]

df_sprzedaz.select(wyrazenia_sum)

suma_sprzedaz_styczen,suma_sprzedaz_luty,suma_sprzedaz_marzec
i64,i64,i64
580,585,615


In [26]:
# ten sam wzorzec przy agregacji grupowej — lista wyrażeń jako argument .agg()
wyrazenia_srednich = [pl.col(kol).mean().round(1).alias(kol) for kol in kolumny_miesiecy]

df_sprzedaz.group_by("region").agg(wyrazenia_srednich)

region,sprzedaz_styczen,sprzedaz_luty,sprzedaz_marzec
str,f64,f64,f64
"""Wschod""",125.0,125.0,140.0
"""Zachod""",165.0,167.5,167.5


Zaleta względem wypisywania kolumn na sztywno: dodanie `sprzedaz_kwiecien`
do danych źródłowych automatycznie obejmuje go w agregacji — zero zmian w
kodzie transformacji, o ile trzyma się tej samej konwencji nazewnictwa.

## 12. Wczytywanie wielu plików — comprehension + `concat`

Częsty wzorzec: folder z plikami CSV/Parquet podzielonymi np. po miesiącach,
które trzeba połączyć w jeden DataFrame.

In [27]:
import io

# symulacja "plików" bez dotykania dysku — w praktyce byłaby to lista Path z .glob("*.csv")
tresci_plikow = [
    "id,wartosc\n1,10\n2,20",
    "id,wartosc\n3,30\n4,40",
    "id,wartosc\n5,50",
]

# jeden wiersz: wczytaj każdy plik do osobnego DataFrame, potem połącz w jeden
df_polaczony = pd.concat([pd.read_csv(io.StringIO(t)) for t in tresci_plikow], ignore_index=True)
df_polaczony

,id,wartosc
0,1,10
1,2,20
2,3,30
3,4,40
4,5,50


Ten sam wzorzec z prawdziwymi plikami wyglądałby tak:

```python
from pathlib import Path

pliki = Path("dane/2025").glob("sprzedaz_*.csv")
df = pd.concat([pd.read_csv(plik) for plik in pliki], ignore_index=True)

# Polars — analogicznie, i szybciej na dużych plikach:
df_pl = pl.concat([pl.read_csv(plik) for plik in pliki])
```

Uwaga wydajnościowa: dla dziesiątek+ plików list comprehension wewnątrz
`pd.concat`/`pl.concat` jest w porządku (samo wczytanie dominuje czas, nie
pętla w Pythonie) — to nie jest przypadek z sekcji 15 (iterowanie po
wierszach), tylko iterowanie po plikach, których jest rzędu dziesiątek/
setek, nie milionów.

## 13. Dict comprehension jako mapowanie kategorii — `.map()` / `.replace()`

Typowy wzorzec: masz krótki słownik reguł biznesowych (np. kod statusu →
czytelna etykieta) i chcesz zastosować go do całej kolumny. Budujesz
słownik comprehension, a samo zastosowanie do kolumny jest już
zwektoryzowane (`.map`), nie pętlą.

In [28]:
df_zamowienia = pd.DataFrame({
    "id_zamowienia": [1, 2, 3, 4],
    "status_kod": ["A", "P", "A", "C"],
})

mapa_statusow_surowa = {"A": "Aktywne", "P": "W realizacji", "C": "Anulowane"}

# dict comprehension przydaje się np. gdy trzeba przefiltrować/zmodyfikować mapowanie przed użyciem —
# tu: wersja z dużymi literami kodu, na wypadek niespójnej wielkości liter w danych źródłowych
mapa_statusow = {kod.upper(): etykieta for kod, etykieta in mapa_statusow_surowa.items()}

df_zamowienia["status_opis"] = df_zamowienia["status_kod"].str.upper().map(mapa_statusow)
df_zamowienia

,id_zamowienia,status_kod,status_opis
0,1,A,Aktywne
1,2,P,W realizacji
2,3,A,Aktywne
3,4,C,Anulowane


## 14. Czyszczenie nazw kolumn — pełny, praktyczny przykład

Złożenie kilku wzorców z tego notebooka w jedną, typową operację
"porządkowania" surowego DataFrame (np. zaraz po wczytaniu z Excela/API).

In [29]:
df_surowy = pd.DataFrame(columns=[
    " ID Klienta ", "Nazwa (pełna)", "E-mail Kontaktowy", "Data Utworzenia  ",
])

df_surowy.columns = [
    kol.strip()                       # usuń białe znaki na brzegach
       .lower()                       # małe litery
       .replace(" ", "_")             # spacje -> podkreślniki
       .replace("(", "").replace(")", "")  # usuń nawiasy
       .replace("-", "_")             # myślniki -> podkreślniki
    for kol in df_surowy.columns
]
print(list(df_surowy.columns))

['id_klienta', 'nazwa_pełna', 'e_mail_kontaktowy', 'data_utworzenia']


## 15. Antywzorzec — comprehension/pętla po WIERSZACH DataFrame

To jest sekcja, którą warto zapamiętać najbardziej. Wszystko powyżej
dotyczyło iterowania po **kolumnach** (zwykle kilka/kilkanaście elementów)
albo **plikach** (dziesiątki) — to tanie. Iterowanie po **wierszach**
DataFrame (tysiące/miliony) to zupełnie inna skala kosztu, bo każda
iteracja w Pythonie ma narzut, którego wektoryzacja (operacje całymi
kolumnami w C/Rust pod spodem) nie ma.

In [30]:
df_duzy = pd.DataFrame({
    "a": np.random.rand(200_000),
    "b": np.random.rand(200_000),
})

# ANTYWZORZEC: list comprehension po wierszach przez .iterrows()
czas_iterrows = timeit.timeit(
    lambda: [row["a"] + row["b"] for _, row in df_duzy.iterrows()],
    number=1,
)

# LEPIEJ: comprehension po itertuples() — szybszy niż iterrows, ale wciąż pętla Pythona
czas_itertuples = timeit.timeit(
    lambda: [t.a + t.b for t in df_duzy.itertuples()],
    number=1,
)

# NAJLEPIEJ: wektoryzacja — cała operacja liczona jednym wywołaniem na poziomie kolumn
czas_wektoryzacja = timeit.timeit(
    lambda: (df_duzy["a"] + df_duzy["b"]).tolist(),
    number=1,
)

print(f"iterrows + comprehension: {czas_iterrows:.4f} s")
print(f"itertuples + comprehension: {czas_itertuples:.4f} s")
print(f"wektoryzacja (a + b):      {czas_wektoryzacja:.4f} s")
print(f"iterrows wolniejsze od wektoryzacji: {czas_iterrows / czas_wektoryzacja:.0f}x")

iterrows + comprehension: 3.6492 s
itertuples + comprehension: 0.0913 s
wektoryzacja (a + b):      0.0060 s
iterrows wolniejsze od wektoryzacji: 610x


`.iterrows()` jest szczególnie kosztowny, bo dla każdego wiersza buduje
osobny obiekt `Series` (z automatyczną konwersją typów). `.itertuples()`
jest wyraźnie szybszy (zwraca lekkie `namedtuple`), więc jeśli **musisz**
iterować po wierszach — używaj `.itertuples()`, nigdy `.iterrows()`. Ale
oba przegrywają z wektoryzacją o rząd wielkości i więcej.

### Kiedy iterowanie po wierszach jest w ogóle uzasadnione?

- Logika naprawdę sekwencyjna, zależna od poprzedniego wiersza w sposób,
  którego nie da się wyrazić przez `.shift()`/`.cumsum()`/`.rolling()`.
- Wywołanie czegoś zewnętrznego per wiersz (request do API, zapis do pliku)
  — tu i tak dominuje koszt I/O, nie pętli w Pythonie.
- Bardzo mała liczba wierszy (dziesiątki/setki) — różnica nieistotna.

W każdym innym przypadku: najpierw szukaj wektorowego odpowiednika
(`.where()`, `np.select()`, `.map()`, operacje na całych kolumnach) zanim
sięgniesz po pętlę czy comprehension po wierszach.

In [31]:
# ten sam wynik co wyżej, ale jako "co zrobić zamiast pętli" — najczęstsze zwektoryzowane odpowiedniki

# zamiast: [x**2 if x > 0.5 else -x for x in df["a"]]
df_duzy["a_transformowana"] = np.where(df_duzy["a"] > 0.5, df_duzy["a"] ** 2, -df_duzy["a"])

# zamiast: dict comprehension + .map() per wiersz w pętli, gdy warunków jest więcej niż jeden
warunki = [df_duzy["a"] > 0.66, df_duzy["a"] > 0.33]
etykiety = ["wysoka", "srednia"]
df_duzy["kategoria"] = np.select(warunki, etykiety, default="niska")

df_duzy.head()

,a,b,a_transformowana,kategoria
0,0.384389,0.374761,-0.384389,srednia
1,0.701054,0.073898,0.491477,wysoka
2,0.234629,0.968210,-0.234629,niska
3,0.819028,0.023502,0.670806,wysoka
4,0.259063,0.547893,-0.259063,niska


## Podsumowanie

| Wzorzec | Kiedy używać |
|---|---|
| `[x for x in ...]` | budowanie listy z transformacją/filtrem — czytelniejsze niż `for` + `.append()` |
| `{k: v for ...}` | budowanie słownika (mapowania, rename, dtype) z reguły, nie ręcznie wypisanych par |
| `{x for ...}` | unikalne wartości przy okazji transformacji |
| `(x for x in ...)` | wynik od razu "konsumowany" przez `sum`/`any`/`max`/`concat` — oszczędza pamięć |
| `:=` w comprehension | filtr i wynik zależą od tego samego kosztownego obliczenia |
| comprehension po **kolumnach/plikach** | tanie — rób to śmiało (sekcje 9-14) |
| comprehension/pętla po **wierszach DataFrame** | ostatnia deska ratunku — najpierw szukaj wektorowego odpowiednika (sekcja 15) |

Najczęstszy błąd początkującego analityka: przenoszenie nawyku "comprehension
jest szybsze i ładniejsze niż pętla" (prawda dla zwykłych list w Pythonie) na
DataFrame, gdzie właściwym porównaniem nie jest "pętla vs comprehension",
tylko "cokolwiek-w-Pythonie vs wektoryzacja".